# 03 - Log Analysis với Drain3

Sau khi metrics chỉ ra `cart-service` là service chính cần điều tra, notebook này phân tích logs để giải thích cơ chế lỗi.

Ý tưởng: raw logs rất nhiều dòng, nên ta dùng Drain3 để gom message thành template. Sau đó đếm template theo thời gian để tìm pattern spike.

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "g2-data" / "g2").exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing g2-data/g2")

ROOT = find_project_root()
DATA = ROOT / "g2-data" / "g2"
LOGS = DATA / "logs"
PLOTS = ROOT / "lab" / "plots"
RESULTS = ROOT / "lab" / "results"
PLOTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def format_time_axis(ax, interval=3):
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    ax.tick_params(axis="x", labelrotation=0, labelsize=8, labelbottom=True)

## 3.1 Đọc log JSONL

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    return df

cart_logs = read_jsonl(LOGS / "cart-service.log.jsonl")
order_logs = read_jsonl(LOGS / "order-service.log.jsonl")

pd.DataFrame([
    {"service": "cart-service", "rows": len(cart_logs), "start": cart_logs["timestamp"].min(), "end": cart_logs["timestamp"].max()},
    {"service": "order-service", "rows": len(order_logs), "start": order_logs["timestamp"].min(), "end": order_logs["timestamp"].max()},
])

## 3.2 Phân bố log level theo thời gian

In [ ]:
cart_logs["hour"] = cart_logs["timestamp"].dt.floor("h")
level_hour = cart_logs.groupby(["hour", "level"]).size().unstack(fill_value=0)
level_hour.tail(8)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
level_hour.plot(kind="area", stacked=True, ax=ax, linewidth=0)
ax.set_title("Cart-service log volume theo level")
ax.set_ylabel("log count/hour")
ax.grid(True, alpha=0.25)
format_time_axis(ax)
fig.tight_layout()
fig.savefig(PLOTS / "03_log_cart_levels_by_hour.png", dpi=160)
plt.show()

**Cart-service log volume theo level**

![Cart-service log volume theo level](../plots/03_log_cart_levels_by_hour.png)

## 3.3 Drain3 template mining

In [ ]:
def parse_with_drain(df):
    config = TemplateMinerConfig()
    config.profiling_enabled = False
    miner = TemplateMiner(config=config)
    rows = []
    for _, event in df.iterrows():
        result = miner.add_log_message(event["message"])
        rows.append({
            "timestamp": event["timestamp"],
            "level": event["level"],
            "service": event["service"],
            "message": event["message"],
            "template": result["template_mined"],
            "cluster_id": result["cluster_id"],
        })
    return pd.DataFrame(rows)

cart_templates = parse_with_drain(cart_logs)
cart_templates["template"].value_counts().head(15)

## 3.4 Template spike quan trọng

In [ ]:
patterns = {
    "cache eviction": "ProductCatalogCache eviction failed",
    "GC warning": "GC overhead limit warning",
    "slow response": "Slow response detected",
    "OOM imminent": "OutOfMemoryError imminent",
    "OOMKilled": "Container OOMKilled",
}

rows = []
for label, needle in patterns.items():
    matched = cart_logs[cart_logs["message"].str.contains(re.escape(needle), regex=True)]
    counts = matched.set_index("timestamp").resample("30min").size()
    rows.append({
        "pattern": label,
        "first_seen": matched["timestamp"].min(),
        "peak_bucket": counts.idxmax(),
        "peak_count_30m": counts.max(),
        "total_count": len(matched),
    })
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for label, needle in patterns.items():
    matched = cart_logs[cart_logs["message"].str.contains(re.escape(needle), regex=True)]
    counts = matched.set_index("timestamp").resample("30min").size()
    ax.plot(counts.index, counts.values, label=label, linewidth=1.4)
ax.set_title("Cart-service log template counts theo bucket 30 phút")
ax.set_ylabel("count/30min")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
format_time_axis(ax)
fig.tight_layout()
fig.savefig(PLOTS / "03_log_cart_template_spikes.png", dpi=160)
plt.show()

**Cart-service log template spikes**

![Cart-service log template spikes](../plots/03_log_cart_template_spikes.png)

## 3.5 Khai thác field phụ: cache_size_mb và heap_used_mb

Ngoài message text, `cart-service.log.jsonl` còn có một số field số rất quan trọng:

- `cache_size_mb`: kích thước cache `ProductCatalogCache`.
- `heap_used_mb`: heap đã dùng khi JVM báo sắp hết bộ nhớ.
- `memory_limit_bytes`: giới hạn memory của container.

Các field này giúp root cause mạnh hơn, vì không chỉ thấy log nói "cache eviction failed", mà còn thấy cache/heap đã lớn tới mức nào.

In [ ]:
numeric_cols = ["timestamp", "level", "message", "cache_size_mb", "heap_used_mb", "memory_limit_bytes"]
numeric = cart_logs[
    cart_logs[["cache_size_mb", "heap_used_mb", "memory_limit_bytes"]].notna().any(axis=1)
][numeric_cols].copy()

summary_rows = []
for field in ["cache_size_mb", "heap_used_mb"]:
    values = numeric.dropna(subset=[field]) if field in numeric else pd.DataFrame()
    if not values.empty:
        summary_rows.append({
            "field": field,
            "first_seen": values["timestamp"].min(),
            "count": len(values),
            "median": values[field].median(),
            "max": values[field].max(),
            "max_at": values.loc[values[field].idxmax(), "timestamp"],
            "example_message": values.loc[values[field].idxmax(), "message"],
        })
pd.DataFrame(summary_rows)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

cache = numeric.dropna(subset=["cache_size_mb"]).set_index("timestamp")["cache_size_mb"]
cache_30m = cache.resample("30min").agg(["median", "max"])
axes[0].plot(cache_30m.index, cache_30m["median"], label="median cache_size_mb", color="#1f77b4")
axes[0].plot(cache_30m.index, cache_30m["max"], label="max cache_size_mb", color="#ff7f0e")
axes[0].axhline(1800, color="#d62728", linestyle="--", linewidth=1, label="~1.8 GB")
axes[0].set_ylabel("cache MB")
axes[0].legend(fontsize=8)

heap = numeric.dropna(subset=["heap_used_mb"]).set_index("timestamp")["heap_used_mb"]
heap_30m = heap.resample("30min").agg(["median", "max"])
axes[1].plot(heap_30m.index, heap_30m["median"], label="median heap_used_mb", color="#9467bd")
axes[1].plot(heap_30m.index, heap_30m["max"], label="max heap_used_mb", color="#d62728")
axes[1].axhline(2048, color="#111111", linestyle="--", linewidth=1, label="2 GB limit")
axes[1].set_ylabel("heap MB")
axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(True, alpha=0.25)
    format_time_axis(ax)
fig.suptitle("Cart-service cache_size_mb và heap_used_mb từ log")
fig.tight_layout()
fig.savefig(PLOTS / "03_log_cart_cache_heap_fields.png", dpi=160)
plt.show()

**Cart-service cache và heap fields**

![Cart-service cache và heap fields](../plots/03_log_cart_cache_heap_fields.png)

Kết quả quan trọng: `cache_size_mb` có lúc lên gần `1.8 GB`, trong khi memory limit của container là khoảng `2 GB`. Khi OOM bắt đầu xuất hiện, `heap_used_mb` nằm khoảng `1.9-1.98 GB`. Đây là bằng chứng rất mạnh cho giả thuyết `ProductCatalogCache` phình lớn, eviction thất bại, rồi dẫn tới memory pressure/OOM.

## 3.6 Order-service log patterns liên quan tới cart

In [ ]:
order_patterns = {
    "cart timeout": "Cart service timeout",
    "cart returned 5xx": "Cart service returned 5xx",
}
rows = []
for label, needle in order_patterns.items():
    matched = order_logs[order_logs["message"].str.contains(re.escape(needle), regex=True)]
    counts = matched.set_index("timestamp").resample("30min").size()
    rows.append({
        "pattern": label,
        "first_seen": matched["timestamp"].min(),
        "first_high_bucket": counts[counts >= 20].index.min(),
        "total_count": len(matched),
    })
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for label, needle in order_patterns.items():
    matched = order_logs[order_logs["message"].str.contains(re.escape(needle), regex=True)]
    counts = matched.set_index("timestamp").resample("30min").size()
    ax.plot(counts.index, counts.values, label=label, linewidth=1.4)
ax.set_title("Order-service log patterns liên quan tới cart-service")
ax.set_ylabel("count/30min")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
format_time_axis(ax)
fig.tight_layout()
fig.savefig(PLOTS / "03_log_order_cart_failures.png", dpi=160)
plt.show()

**Order-service patterns liên quan tới cart**

![Order-service patterns liên quan tới cart](../plots/03_log_order_cart_failures.png)

Log analysis giải thích cơ chế lỗi: cache eviction failure và GC warning spike từ `14:00Z`, sau đó `OutOfMemoryError imminent` và `OOMKilled` xuất hiện lúc `19:59Z`. Đây là evidence trực tiếp cho giả thuyết memory pressure/restart loop.